# 03 | Exploratory Analysis: Merging Panels and Visual Inspection

This notebook merges the IEA spending panel with the OpenAlex publications panel
on (country, technology, year), produces a dual-axis time-series grid for visual
inspection, and computes raw-level cross-correlation functions as a first
(naïve) look at lead/lag structure.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

spend = pd.read_csv("../data/processed/rdd_public_panel.csv")
pubs  = pd.read_csv("../data/processed/openalex_full_panel.csv")


## Country code reconciliation

In [ ]:
# IEA uses ISO-3 names; OpenAlex returns ISO-2 codes. Map ISO-2 → ISO-3 country names.
ISO2_TO_NAME = {
    "US": "United States", "GB": "United Kingdom", "DE": "Germany", "FR": "France",
    "JP": "Japan", "IT": "Italy", "CA": "Canada", "AU": "Australia", "NL": "Netherlands",
    "ES": "Spain", "SE": "Sweden", "NO": "Norway", "FI": "Finland", "DK": "Denmark",
    "BE": "Belgium", "AT": "Austria", "CH": "Switzerland", "IE": "Ireland",
    "PT": "Portugal", "GR": "Greece", "PL": "Poland", "CZ": "Czechia",
    "HU": "Hungary", "SK": "Slovakia", "TR": "Turkey", "MX": "Mexico",
    "KR": "Korea", "NZ": "New Zealand", "EE": "Estonia", "LV": "Latvia",
    "LT": "Lithuania", "SI": "Slovenia", "LU": "Luxembourg", "IS": "Iceland",
}
pubs["country"] = pubs["country"].map(ISO2_TO_NAME)
pubs = pubs.dropna(subset=["country"])


## Inner merge

In [ ]:
merged = spend.merge(pubs, on=["country", "technology", "year"], how="inner")
print(f"merged rows: {len(merged):,}")
print(f"countries: {merged['country'].nunique()}; technologies: {merged['technology'].nunique()}")
merged.to_csv("../data/processed/merged_panel.csv", index=False)


## Dual-axis time series, 9 panels, OECD aggregate

In [ ]:
TECHS = sorted(merged["technology"].unique())
agg = merged.groupby(["technology", "year"], as_index=False).agg(
    spending=("spending_usd_ppp_millions", "sum"),
    pubs=("pub_count", "sum"),
)

fig, axes = plt.subplots(3, 3, figsize=(15, 12), sharex=True)
for ax, tech in zip(axes.flat, TECHS):
    sub = agg.query("technology == @tech")
    ax2 = ax.twinx()
    ax.plot(sub["year"], sub["spending"], color="C0", label="Spending")
    ax2.plot(sub["year"], sub["pubs"], color="C3", label="Publications")
    ax.set_title(tech, fontsize=10)
    ax.tick_params(axis="y", labelcolor="C0")
    ax2.tick_params(axis="y", labelcolor="C3")
fig.suptitle("Public R&D spending (blue) vs publications (red), OECD aggregate", fontsize=13)
plt.tight_layout()
plt.savefig("../figures/dual_axis_grid.png", dpi=150, bbox_inches="tight")
plt.show()


## Cross-correlation on raw levels, caveat

Both series are non-stationary (trends), so raw-level CCFs are dominated by
shared trend rather than true lead/lag structure. We run them anyway as
exposition; notebook 04 redoes this on first-differenced series.


In [ ]:
def ccf(x, y, max_lag=10):
    x = np.asarray(x); y = np.asarray(y)
    x = (x - x.mean()) / x.std(); y = (y - y.mean()) / y.std()
    n = len(x)
    return [np.sum(x[:n-k] * y[k:]) / n for k in range(-max_lag, max_lag + 1)]

LAGS = list(range(-10, 11))
fig, axes = plt.subplots(3, 3, figsize=(13, 10), sharex=True)
for ax, tech in zip(axes.flat, TECHS):
    sub = agg.query("technology == @tech").sort_values("year")
    if len(sub) < 25:
        ax.set_title(f"{tech} (insufficient)"); continue
    cc = ccf(sub["spending"], sub["pubs"])
    ax.bar(LAGS, cc, color="grey")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_title(tech, fontsize=10)
fig.suptitle("Raw-level CCF: spending vs publications (lag in years)", fontsize=12)
plt.tight_layout()
plt.savefig("../figures/ccf_grid.png", dpi=150, bbox_inches="tight")
plt.show()


## Output

- `../data/processed/merged_panel.csv`, merged spending + publications panel.
- `../figures/dual_axis_grid.png`, visual co-movement check.
- `../figures/ccf_grid.png`, raw-level CCFs (naïve, dominated by trends).

Visual inspection shows clear co-movement between spending and publications for
nuclear (declining together) and hydrogen (rising together). Trend dominance
motivates the first-differenced approach in notebook 04.
